In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

In [ ]:
from lib import genius
mio   = genius.MusicDBIO(verbose=False)
apiio = genius.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

In [ ]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
mediaData          = MusicDBData(path=permDir, fname="{0}MediaData".format(db.lower()))
knownSongs         = MusicDBData(path=permDir, fname="{0}SongsData".format(db.lower()))
localSongs         = MusicDBData(path=permDir, fname="{0}SearchedForLocalSongs".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [ ]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
#print("   Known Songs Search:        {0}".format(len(knownSongs.get())))
print("   Local Songs Search:        {0}".format(len(localSongs.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

# Search For New Artists

In [ ]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = genius.RawAPIData(debug=False)

## Find Artists To Get

In [ ]:
useMasterData = True

if useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["Genius"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  624571
#   Artist Names To Get:  614475
#   Artist Names To Get:  599423

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchData(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)


In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Data

In [ ]:
from pandas import DataFrame, Series, concat

def getArtistNamesSeries(mad):
    s = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,dict):
                searchData.update({str(int(k)): v for k,v in searchTermData.items() if k is not None})
        s = Series(searchData)
    return s
        
def getResponseSeries(mad):
    s = getArtistNamesSeries(mad)
    if not isinstance(s,Series):
        return None
    return s

In [ ]:
mad = masterArtistsData.get()
s   = getResponseSeries(mad)
if isinstance(s,Series):
    print("Found {0} New Artists".format(len(s)))
    searchDF = genius.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(len(searchDF)))
        searchDF = concat([searchDF, s])
    else:
        print("Found 0 Previous Artists")
        searchDF = s
    print("Found {0} Total Artists".format(len(searchDF)))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists Searches".format(len(searchDF)))
    print("  ==> {0} Old Artists".format(len(searchDF[searchDF.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])))
    print("Saving Data")
    genius.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Info

In [ ]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = genius.RawAPIData(debug=False)

## Find Artists To Get

In [ ]:
artistNames       = genius.MusicDBIO(local=False).data.getSearchArtistData()
localArtistsDict  = localArtists.get()
artistIDsToGet    = artistNames[~artistNames.index.isin(localArtistsDict.keys())]

print("{0} Search Results".format(db))
print("   Available Names:      {0}".format(len(artistNames)))
print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("   Artist Names To Get:  {0}".format(len(artistIDsToGet)))
    
#   Artist Names To Get:  65042
#   Artist Names To Get:  58441
#   Artist Names To Get:  50441
#   Artist Names To Get:  29466
#   Artist Names To Get:  18341

## Download Artist Infos

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "9:15pm")
tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:00am")
n  = 0
localArtistsDict = localArtists.get()
searchedForErrors = errors.get()
for i,(dbID,artistName) in enumerate(artistIDsToGet.iteritems()):    
    if localArtistsDict.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistInfo(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)        
    localArtistsDict[dbID] = artistName
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artists".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

# Download Album Data

In [ ]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = genius.RawAPIData(debug=True)

## Find Albums To Get

In [ ]:
useKnownArtists  = False
useSearchArtists = True

if useKnownArtists is True:
    print("Genius Search Results")
    print("   Known Summary IDs:    {0}".format(len(knownArtists)))
    artistNamesToGet = knownArtists[~knownArtists.index.isin(localAlbums.get().keys())]
    artistNamesToGet = artistNamesToGet.sample(frac=1)
    print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))
elif useSearchArtists is True:
    print("Genius Search Results")
    artistNames       = genius.MusicDBIO(local=False).data.getSearchArtistData()
    print("   Known Search IDs:      {0}".format(len(artistNames)))
    localAlbumsDict  = localAlbums.get()
    print("   Downloaded Albums IDs: {0}".format(len(localAlbumsDict)))
    artistNamesToGet    = artistNames[~artistNames.index.isin(localAlbumsDict.keys())]
    print("   Artists IDs To Get:    {0}".format(len(artistNamesToGet)))
    
    
#   Artists IDs To Get:   65089
#   Artists IDs To Get:    61364
#   Artists IDs To Get:    50539
#   Artists IDs To Get:    50039
#   Artists IDs To Get:    35589
#   Artists IDs To Get:    14289

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "9:15pm")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "11:00am")
n  = 0
searchedForAlbums = localAlbums.get()
searchedForErrors = errors.get()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistSongs(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        localAlbums.save(data=searchedForAlbums)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
localAlbums.save(data=searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
del searchedForErrors['364661']
errors.save(data=searchedForErrors)

# Download Song Data

In [ ]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = genius.RawAPIData(debug=True)

## Create Media Data List

In [ ]:
from pandas import Series, DataFrame
mediaDataSeries = None
mdbio   = genius.MusicDBIO(local=False)
for modVal in range(100):
    modValData = mdbio.data.getModValData(modVal)
    modValMediaData = modValData.apply(lambda rData: {mediaType: {record.code: record.album for record in media} for mediaType,media in rData.media.media.items()})
    mediaDataSeries = concat([mediaDataSeries, modValMediaData]) if isinstance(mediaDataSeries,Series) else modValMediaData
    if (modVal+1) % 10 == 0:
        print(f"{modVal+1}  |  {len(mediaDataSeries)}")
mediaData.save(data=mediaDataSeries)

In [ ]:
mediaDataSeries = mediaData.get()
mediaDataSeries.name = "Songs"
knownSongsData = []
from timeutils import Timestat
ts = Timestat("Creating Master Songs Index")
N = len(mediaDataSeries)
for i,(artistID,artistIDMedia) in enumerate(mediaDataSeries.iteritems()):
    songs = artistIDMedia.get('Song', {})
    knownSongsData += list(zip([artistID]*len(songs), songs.keys(), songs.values()))
    if (i+1) % 50000 == 0:
        ts.update(n=i+1, N=N, cmt=f"Nsongs={len(knownSongsData)}")
ts.stop()

ts = Timestat(f"Saving {N} Song Indices")
df = DataFrame(knownSongsData)
df.columns = ["ArtistID", "SongID", "SongName"]
df["ArtistName"] = df["ArtistID"].map(knownArtists)
df = df[~df.duplicated(subset=["SongID", "SongName"])]
df.index = df["SongID"]
df = df.drop(["SongID"], axis=1)
N = df.shape[0]
knownSongs.save(data=df)
ts.stop()

## Find Songs To Download

In [ ]:
print("# Genius Song Search Results")
knownSongsData = knownSongs.get()
print("#   Available Song IDs:    {0}".format(knownSongsData.shape[0]))
localSongsData = localSongs.get()
print("#   Known Song IDs:        {0}".format(len(localSongsData)))
artistNamesToGet = knownSongsData[~knownSongsData.index.isin(localSongsData.keys())].sample(frac=1)
print("#   Songs To Download:     {0}".format(artistNamesToGet.shape[0]))

#   Songs To Download:     4708473

## Download Songs

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Songs".format(db))
#tt = TermTime("tomorrow", "9:50am")
tt = TermTime("today", "7:00pm")
n  = 0
searchedForSongs = localSongs.get()
searchedForErrors = errors.get()
for i,(songID,row) in enumerate(artistNamesToGet.iterrows()):
    artistID   = row["ArtistID"]
    artistName = row["ArtistName"]
    songName   = row["SongName"]
    if searchedForSongs.get(songID) is not None:
        continue
    if searchedForErrors.get(songID) is not None:
        continue
        
    modVal=mio.mv.get(songID)
    if mio.data.getRawArtistSongFilename(modVal, songID).exists():
        searchedForSongs[songID] = (artistName,songName)
        continue
    
    response = apiio.getArtistSongData(artistName=artistName, songName=songName, songID=songID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((songID,artistName)))
        searchedForErrors[songID] = (artistName,songName)
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistSongData(data=response, modval=modVal, dbID=songID)
    searchedForSongs[songID] = (artistName,songName)
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(5)
        
    if n % 100 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Songs".format(len(searchedForSongs), db))
        localSongs.save(data=searchedForSongs)
        print("="*150)
        apiio.wait(20.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Songs".format(len(searchedForSongs), db))
localSongs.save(data=searchedForSongs)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
del searchedForErrors['1505302']
errors.save(data=searchedForErrors)

In [ ]:
mio.data.saveRawArtistSongData(data=response, modval=modVal, dbID=songID)
searchedForSongs[songID] = (artistName,songName)
print("Saving {0} {1} Searched For Songs".format(len(searchedForSongs), db))
localSongs.save(data=searchedForSongs)

# Move Local Files

In [ ]:
from lib.genius import moveLocalFiles
moveLocalFiles()

In [ ]:
from fileutils import FileInfo
from mdbmaster import MasterParams
mp        = MasterParams()
mioLocal  = genius.MusicDBIO(local=True,mkDirs=True,debug=True)
mioGlobal = genius.MusicDBIO(local=False,mkDirs=True,debug=True)
for modVal in mp.getModVals():
    files = mioLocal.dir.getRawAlbumModValDataDir(modVal).glob("*.p")
    _ = [FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistAlbumFilename(modVal,ifile.stem))) for ifile in files]

In [ ]:
io.get("/Volumes/Piggy/Discog/artists-genius/74/1010174.p")

In [ ]:
""" MusicDBIO Utilities """

__all__ = ["moveLocalFiles"]

from fileutils import FileInfo
from master import MasterParams
#from .musicdbio import MusicDBIO
from lib.genius import MusicDBIO

def moveLocalFiles(**kwargs):    
    verbose = kwargs.get('verbose')
    if verbose: print("moveLocalFiles()")
    mp        = MasterParams()
    mioLocal  = MusicDBIO(local=True,mkDirs=True,debug=verbose)
    mioGlobal = MusicDBIO(local=False,mkDirs=True,debug=verbose)
    for modVal in mp.getModVals():
        ## Artist 
        files = mioLocal.dir.getRawModValDataDir(modVal).glob("*.p")
        _ = [FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistInfoFilename(modVal,ifile.stem))) for ifile in files]

In [ ]:
moveLocalFiles()

In [ ]:
mioLocal  = MusicDBIO(local=True, mkDirs=False)
mioLocal.data.getRawArtistInfoFilename(0, 0).str

# API Data

In [ ]:
from lib.genius import MusicDBIO
mdbio = MusicDBIO(verbose=True)

In [ ]:
mdbio.prd.parseArtistData(0, force=True)

In [ ]:
mdbio.data.getModValArtistData(0)['70000'].show()

In [ ]:
search_term = "Missy Elliott"

In [ ]:
client_access_token = "bHPww2B5V9jYrR0Hshp4GjSUdOR2Deu5uanDGVzvOFc0ElvWIjYp_YSXmnnJdhS1"
baseURL = "https://api.genius.com"
song_id='378195'
apipath=f"{baseURL}/songs/{song_id}?access_token={client_access_token}"
#album_id='750897'
#apipath=f"{baseURL}/albums/{album_id}&access_token={apikey}"

In [ ]:
from apiutils import APIIO
from lib.genius import RawAPIData
apiio = RawAPIData()

In [ ]:
apiio = APIIO("Genius")
print(apipath)
retval = apiio.get(apipath)

In [ ]:
retval['response']['song']['custom_performances']

In [ ]:
GeniusSongRecord(retval['response']['song']).get()

In [ ]:
https://api.genius.com/oauth/authorize?
client_id=YOUR_CLIENT_ID&
redirect_uri=YOUR_REDIRECT_URI&
scope=REQUESTED_SCOPE&
state=SOME_STATE_VALUE&
response_type=code

In [ ]:
response = apiio.get(apipath)
results  = apiio.getResponse(response)
#apiio.getResponse()

In [ ]:
apipath

In [ ]:
stop = 5000
dbIO = dbGeniusAPI()
api  = geniusAPIIO()
ts   = timestat("Getting Artist Data From Genius API")
n    = 0

In [ ]:
from glob import glob
from masterUtils import masterUtils
from fsUtils import dirUtil
mu = masterUtils()
artistsDir = dirUtil(mu.getDisc("GeniusAPI").getArtistsDir())
#### This takes forever...
#geniusArtistFiles = {modVal: glob(str(dirUtil(artistsDir.join(str(modVal))).join("*.p"))) for modVal in range(100)}